In [0]:
from pyspark.sql.functions import col, count, when, isnan, isnull, countDistinct, lit, trim, year, month, current_timestamp
from pyspark.errors import AnalysisException
import re
import pandas as pd

In [0]:

CATALOG    = "project"
SCHEMA     = "cambridgeshire_data_raw"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/volume"

#Get all table names in the schema (filter to only street crime tables)
tables = [
    row.tableName 
    for row in spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA}").collect()
    if 'street' in row.tableName.lower()  # Only include street crime tables
]

print(f"Found {len(tables)} tables in {CATALOG}.{SCHEMA}")

#Iteratively ingest each table
dfs = []

for table in tables:
    full_table_name = f"{CATALOG}.{SCHEMA}.{table}"
    try:
        df_spark = (
            spark.table(full_table_name)
                .withColumn("source_table", lit(table))
                .withColumn("ingested_at", current_timestamp())
        )
        dfs.append(df_spark)
        print(f"✓ Loaded: {table}")

    except AnalysisException as e:
        print(f"Table not found: {table} — {str(e)}")

In [0]:
# Combine all DataFrames into a single DataFrame
from functools import reduce

if dfs:
    # Union all DataFrames using unionByName to handle any column order differences
    df = reduce(lambda df1, df2: df1.unionByName(df2, allowMissingColumns=True), dfs)
    
    print(f"Successfully combined {len(dfs)} tables")
    print(f"Total rows: {df.count():,}")
    print(f"Total columns: {len(df.columns)}")
    print(f"\nColumns: {', '.join(df.columns)}")
else:
    print("No DataFrames to combine")
    df = None

if df is None:
    raise ValueError("DataFrame is empty — no tables were successfully loaded.")

In [0]:
# Convert column headers to snake_case

def to_snake_case(name):
    """Converts a string to snake_case"""
    name = name.replace(' ', '_')
    name = name.lower()
    return name

print("Column name conversions:")
for col_name in df.columns:
    new_name = to_snake_case(col_name)
    if col_name != new_name:
        print(f"{col_name} changed to {new_name}")
        df = df.withColumnRenamed(col_name, new_name)
    else:
        print(f"{col_name} (already snake_case)")

Validatin

In [0]:
# Count rows in df and all individual tables

# Count rows in combined df
df_count = df.count()

# Count rows in each individual table
print("Individual table row counts: \n")

table_counts = []
for table in tables:
    count = spark.table(f"project.cambridgeshire_data_raw.{table}").count()
    table_counts.append((table, count))
    print(f"{table:40} {count:>8,}")

# Calculate total from individual tables
total_individual = sum(count for _, count in table_counts)
print("\n")
print(f"{'Total from individual tables:':40} {total_individual:>8,}")
print(f"{'Combined df count:':40} {df_count:>8,}")
print(f"{'Match:':40} {df_count == total_individual}")

In [0]:
# Initial Overall Data Quality Assessment
# 1. Missing Values
print("MISSING VALUES")
missing_data = []
for column in df.columns:
    null_count = df.filter(col(column).isNull()).count()
    null_pct = (null_count / df.count()) * 100
    missing_data.append((column, null_count, null_pct))
    print(f"{column:30} {null_count:>10,} ({null_pct:>6.2f}%)")

# 2. Duplicate check
print("\n DUPLICATE RECORDS")
total_rows = df.count()
distinct_rows = df.distinct().count()
duplicates = total_rows - distinct_rows
print(f"Total Rows: {total_rows:,}")
print(f"Distinct Rows: {distinct_rows:,}")
print(f"Duplicate Rows: {duplicates:,}")

# Check Crime ID duplicates - primary key so (non-null) duplicates here indicate the row is an actual duplicate.
crime_id_total = df.filter(col("Crime ID").isNotNull()).count()
crime_id_distinct = df.filter(col("Crime ID").isNotNull()).select("crime_id").distinct().count()
print(f"\nCrime IDs (non-null): {crime_id_total:,}")
print(f"Distinct Crime IDs: {crime_id_distinct:,}")
print(f"Duplicate Crime IDs: {crime_id_total - crime_id_distinct:,}")

# 3. Geographic Data Null Check
print("\nGEOGRAPHIC DATA NULLS")
geo_null = df.filter(col("longitude").isNull() | col("latitude").isNull() | col("lsoa_code").isNull() | col("lsoa_name").isNull() | col("location").isNull()).count()
print(f"Records without geographical data: {geo_null:,} ({(geo_null/total_rows)*100:.2f}%)")

# 4. Categorical Fields Summary
print("\nCATEGORICAL FIELDS SUMMARY")
print(f"Unique Crime Types: {df.select('crime_type').distinct().count()}")
print(f"Unique Outcome Categories: {df.select('last_outcome_category').distinct().count()}")
print(f"Unique Locations: {df.select('location').distinct().count():,}")
print(f"Unique LSOA Codes: {df.select('lsoa_code').distinct().count()}")
print(f"Unique LSOA Names: {df.select('lsoa_name').distinct().count()}")

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("project.cambridgeshire_combined.combined_raw")